In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", None)

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_HOURLY = BASE_DIR / "data" / "hourly" / "bizi_hourly.csv"


In [ ]:
df = pd.read_csv(DATA_HOURLY, parse_dates=["timestamp"])
df.info()
df.head()


In [ ]:
print("Estaciones únicas:", df["station_id"].nunique())
print("Registros totales:", len(df))

df.describe(include="all")


In [ ]:
df_global = (
    df.groupby("timestamp")
      .agg(
          bikes_totales=("bikes", "sum"),
          slots_totales=("slots", "sum"),
          ratio_medio=("ratio_ocupacion", "mean")
      )
      .reset_index()
)

plt.figure(figsize=(12,5))
plt.plot(df_global["timestamp"], df_global["bikes_totales"], marker="o")
plt.title("Evolución global de bicis disponibles")
plt.xlabel("Tiempo")
plt.ylabel("Bicis totales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
station = df["station_id"].iloc[0]  # puedes cambiarla
df_est = df[df["station_id"] == station].sort_values("timestamp")

plt.figure(figsize=(12,4))
plt.plot(df_est["timestamp"], df_est["bikes"], marker="o")
plt.title(f"Evolución estación {station} - {df_est['station_name'].iloc[0]}")
plt.xlabel("Tiempo")
plt.ylabel("Bicis disponibles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
df["hora"] = df["timestamp"].dt.hour

df_hora = (
    df.groupby("hora")
      .agg(ratio_medio=("ratio_ocupacion", "mean"))
      .reset_index()
)

plt.figure(figsize=(10,4))
sns.lineplot(data=df_hora, x="hora", y="ratio_medio", marker="o")
plt.title("Ratio medio por hora del día")
plt.xlabel("Hora")
plt.ylabel("Ratio medio")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
df_estaciones = (
    df.groupby(["station_id", "station_name"])
      .agg(
          ratio_medio=("ratio_ocupacion", "mean"),
          bikes_medias=("bikes", "mean"),
          n_registros=("timestamp", "count")
      )
      .reset_index()
)

top_altas = df_estaciones.sort_values("ratio_medio", ascending=False).head(10)
top_bajas = df_estaciones.sort_values("ratio_medio", ascending=True).head(10)

top_altas, top_bajas


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(
    data=top_altas,
    x="ratio_medio",
    y="station_name",
    palette="Reds_r"
)
plt.title("Top 10 estaciones con mayor ratio medio de ocupación")
plt.xlabel("Ratio medio")
plt.ylabel("Estación")
plt.tight_layout()
plt.show()
